In [ ]:
# ============================================================
# CELL 1 — Imports  (no new installs needed — all already in venv)
# ============================================================
import os, glob, shutil, random, math
import cv2
import numpy as np
import yaml
import matplotlib.pyplot as plt
from collections import Counter
from ultralytics import YOLO

print('All imports OK')
print('OpenCV:', cv2.__version__)
import ultralytics; print('Ultralytics:', ultralytics.__version__)

In [ ]:
# ============================================================
# CELL 2 — Configuration
# ============================================================

# Class mapping — folder name in sample_images/ -> class ID
CLASS_MAP = {
    'Bull':     0,
    'Buffallo': 1,   # note: folder is spelled 'Buffallo'
    'Lion':     2,
    'Tiger':    3,
    'Elephant': 4,
    'zebra':    5,
    'panda':    6,
}
YOLO_NAMES = {0:'bull', 1:'buffalo', 2:'lion', 3:'tiger',
               4:'elephant', 5:'zebra', 6:'panda'}

SOURCE_DIR  = 'sample_images'    # your existing images
DATASET_DIR = 'aug_dataset'      # augmented dataset output
TRAIN_NAME  = 'wildlife_v2'
MODEL_BASE  = 'yolov8s.pt'       # small — much better than nano
EPOCHS      = 80
IMG_SIZE    = 640
BATCH       = 8                  # lower batch for stability
AUG_PER_IMAGE = 9               # generate 9 augmented copies per original = ~10x dataset

CONF_LABEL  = 0.20              # YOLO-World confidence for auto-labeling

CLASS_COLORS = [
    (0,220,0),(255,140,0),(0,160,255),
    (255,0,255),(255,220,0),(0,255,200),(160,0,255)
]

print('Classes:', YOLO_NAMES)
print(f'Augmentation: {AUG_PER_IMAGE}x per image')
print(f'Model: {MODEL_BASE}, Epochs: {EPOCHS}')

In [ ]:
# ============================================================
# CELL 3 — Auto-label with YOLO-World (teacher model)
# ============================================================
# YOLO-World is a zero-shot open-vocab detector.
# We use it to generate bounding box labels for our images.
# Fallback: if YOLO-World misses, use 70% centre-crop box.

print('Loading YOLO-World teacher model...')
teacher = YOLO('yolov8s-world.pt')
teacher.set_classes(['bull', 'buffalo', 'lion', 'tiger',
                     'elephant', 'zebra', 'giant panda'])

# Reverse map: world class name -> our YOLO id
WORLD_TO_ID = {
    'bull':0, 'buffalo':1, 'lion':2, 'tiger':3,
    'elephant':4, 'zebra':5, 'giant panda':6
}

os.makedirs(os.path.join(DATASET_DIR,'images','train'), exist_ok=True)
os.makedirs(os.path.join(DATASET_DIR,'images','val'),   exist_ok=True)
os.makedirs(os.path.join(DATASET_DIR,'labels','train'), exist_ok=True)
os.makedirs(os.path.join(DATASET_DIR,'labels','val'),   exist_ok=True)

RAW_IMG_DIR = os.path.join(DATASET_DIR, 'raw_labeled')
RAW_LBL_DIR = os.path.join(DATASET_DIR, 'raw_labels')
os.makedirs(RAW_IMG_DIR, exist_ok=True)
os.makedirs(RAW_LBL_DIR, exist_ok=True)

total_labeled = 0
fallback_count = 0

for folder_name, cls_id in CLASS_MAP.items():
    folder_path = os.path.join(SOURCE_DIR, folder_name)
    if not os.path.isdir(folder_path):
        print(f'  WARNING: folder not found: {folder_path}')
        continue

    images = []
    for ext in ('*.jpg','*.jpeg','*.png','*.webp'):
        images.extend(glob.glob(os.path.join(folder_path, ext)))

    print(f'  {folder_name} (cls {cls_id}): {len(images)} images', end=' ... ')

    for i, img_path in enumerate(images):
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        results = teacher(img, conf=CONF_LABEL, verbose=False)
        boxes_yolo = []

        for box in results[0].boxes:
            det_cls = results[0].names[int(box.cls[0])]
            det_id  = WORLD_TO_ID.get(det_cls, -1)
            if det_id != cls_id:   # only keep boxes matching the correct class
                continue
            x_c, y_c, bw, bh = box.xywhn[0].tolist()
            x_c = max(0.001, min(0.999, x_c))
            y_c = max(0.001, min(0.999, y_c))
            bw  = max(0.01,  min(0.999, bw))
            bh  = max(0.01,  min(0.999, bh))
            boxes_yolo.append(f'{cls_id} {x_c:.6f} {y_c:.6f} {bw:.6f} {bh:.6f}')

        if not boxes_yolo:
            # Fallback: 70% centre box (animal is usually centred)
            boxes_yolo.append(f'{cls_id} 0.5 0.5 0.70 0.70')
            fallback_count += 1

        stem = f'{folder_name}_{i}'
        cv2.imwrite(os.path.join(RAW_IMG_DIR, stem+'.jpg'), img)
        with open(os.path.join(RAW_LBL_DIR, stem+'.txt'), 'w') as f:
            f.write('\n'.join(boxes_yolo))
        total_labeled += 1

    print('done')

print(f'\nLabeled {total_labeled} images ({fallback_count} used fallback centre-box)')

In [ ]:
# ============================================================
# CELL 4 — Augment with pure OpenCV/NumPy (10x dataset expansion)
# ============================================================
# Augmentations applied:
#   horizontal flip, brightness/contrast, HSV jitter,
#   rotation, Gaussian blur, mosaic-style crop

def aug_flip(img, boxes):
    img_f = cv2.flip(img, 1)
    new_boxes = []
    for b in boxes:
        c, cx, cy, bw, bh = b
        new_boxes.append((c, 1.0-cx, cy, bw, bh))
    return img_f, new_boxes

def aug_brightness(img, boxes, factor=None):
    f = factor if factor else random.uniform(0.5, 1.5)
    img_b = np.clip(img.astype(np.float32) * f, 0, 255).astype(np.uint8)
    return img_b, boxes

def aug_hsv(img, boxes):
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    img_hsv[:,:,0] = (img_hsv[:,:,0] + random.uniform(-18, 18)) % 180
    img_hsv[:,:,1] = np.clip(img_hsv[:,:,1] * random.uniform(0.7, 1.3), 0, 255)
    img_hsv[:,:,2] = np.clip(img_hsv[:,:,2] * random.uniform(0.7, 1.3), 0, 255)
    return cv2.cvtColor(img_hsv.astype(np.uint8), cv2.COLOR_HSV2BGR), boxes

def aug_blur(img, boxes):
    k = random.choice([3,5])
    return cv2.GaussianBlur(img,(k,k),0), boxes

def aug_rotate(img, boxes, angle=None):
    a = angle if angle else random.uniform(-15, 15)
    h,w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2,h/2), a, 1.0)
    img_r = cv2.warpAffine(img, M, (w,h), borderValue=(114,114,114))
    # Rotate boxes (approximate: just clamp, sufficient for small angles)
    return img_r, boxes

def aug_scale_crop(img, boxes, scale=None):
    s = scale if scale else random.uniform(0.6, 0.95)
    h,w = img.shape[:2]
    new_h, new_w = int(h*s), int(w*s)
    top  = random.randint(0, h-new_h)
    left = random.randint(0, w-new_w)
    img_c = img[top:top+new_h, left:left+new_w]
    img_c = cv2.resize(img_c, (w,h))
    # Adjust box coordinates
    new_boxes = []
    for (c, cx, cy, bw, bh) in boxes:
        # Convert to pixel, adjust, convert back
        px = cx*w; py = cy*h; pw = bw*w; ph = bh*h
        px = (px-left)/s; py = (py-top)/s
        pw = pw/s; ph = ph/s
        ncx = px/w; ncy = py/h; nbw = pw/w; nbh = ph/h
        ncx = max(0.02, min(0.98, ncx)); ncy = max(0.02, min(0.98, ncy))
        nbw = max(0.02, min(0.96, nbw)); nbh = max(0.02, min(0.96, nbh))
        new_boxes.append((c, ncx, ncy, nbw, nbh))
    return img_c, new_boxes

AUG_FNS = [aug_flip, aug_brightness, aug_hsv, aug_blur, aug_rotate, aug_scale_crop]

raw_imgs = sorted(glob.glob(os.path.join(RAW_IMG_DIR, '*.jpg')))
print(f'Augmenting {len(raw_imgs)} labeled images x{AUG_PER_IMAGE}...')

all_for_split = []   # (img_path, lbl_path) of ALL augmented images

for img_path in raw_imgs:
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(RAW_LBL_DIR, stem+'.txt')

    img = cv2.imread(img_path)
    if img is None:
        continue
    with open(lbl_path) as f:
        raw_lines = [l.strip() for l in f if l.strip()]
    boxes = [tuple([int(p) if i==0 else float(p)
                    for i,p in enumerate(l.split())]) for l in raw_lines]

    # Save original
    out_img = os.path.join(DATASET_DIR, 'all_images', f'{stem}_orig.jpg')
    out_lbl = os.path.join(DATASET_DIR, 'all_labels', f'{stem}_orig.txt')
    os.makedirs(os.path.dirname(out_img), exist_ok=True)
    os.makedirs(os.path.dirname(out_lbl), exist_ok=True)
    cv2.imwrite(out_img, img)
    shutil.copy2(lbl_path, out_lbl)
    all_for_split.append((out_img, out_lbl))

    # Generate augmented versions
    for aug_i in range(AUG_PER_IMAGE):
        a_img, a_boxes = img.copy(), list(boxes)
        # Apply 1-3 random augmentations
        chosen = random.sample(AUG_FNS, k=random.randint(1,3))
        for fn in chosen:
            try:
                a_img, a_boxes = fn(a_img, a_boxes)
            except Exception:
                pass

        out_img = os.path.join(DATASET_DIR, 'all_images', f'{stem}_aug{aug_i}.jpg')
        out_lbl = os.path.join(DATASET_DIR, 'all_labels', f'{stem}_aug{aug_i}.txt')
        cv2.imwrite(out_img, a_img)
        with open(out_lbl,'w') as f:
            for b in a_boxes:
                f.write(f'{int(b[0])} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n')
        all_for_split.append((out_img, out_lbl))

print(f'Generated {len(all_for_split)} total images (original + augmented)')

In [ ]:
# ============================================================
# CELL 5 — Train/val split  +  write data.yaml
# ============================================================

random.shuffle(all_for_split)
split_idx = int(len(all_for_split) * 0.85)  # 85% train, 15% val
train_set = all_for_split[:split_idx]
val_set   = all_for_split[split_idx:]

def copy_split(pairs, split_name):
    for img_src, lbl_src in pairs:
        fname = os.path.basename(img_src)
        shutil.copy2(img_src, os.path.join(DATASET_DIR,'images', split_name, fname))
        shutil.copy2(lbl_src, os.path.join(DATASET_DIR,'labels', split_name,
                                           os.path.splitext(fname)[0]+'.txt'))

copy_split(train_set, 'train')
copy_split(val_set,   'val')
print(f'Train: {len(train_set)} | Val: {len(val_set)}')

# Class distribution
cls_count = Counter()
for _, lbl in train_set:
    with open(lbl) as f:
        for line in f:
            if line.strip():
                cls_count[int(line.strip().split()[0])] += 1
print('\nClass distribution in train (box count):')
for i, name in YOLO_NAMES.items():
    bar = '#' * min(cls_count[i]//20, 40)
    print(f'  {i} {name:<10} {cls_count[i]:>5}  {bar}')

data_yaml_path = os.path.join(DATASET_DIR, 'data.yaml')
data_cfg = {
    'path': os.path.abspath(DATASET_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'nc':    7,
    'names': YOLO_NAMES,
}
with open(data_yaml_path,'w') as f:
    yaml.dump(data_cfg, f, sort_keys=False)
print(f'\ndata.yaml: {data_yaml_path}')

In [ ]:
# ============================================================
# CELL 6 — Train YOLOv8s  (~10-60 min depending on GPU)
# ============================================================

model = YOLO(MODEL_BASE)
print(f'Training {MODEL_BASE} for {EPOCHS} epochs on {len(train_set)} images...')

results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    mosaic=1.0,
    close_mosaic=10,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    project='runs/detect',
    name=TRAIN_NAME,
    exist_ok=True,
    verbose=True,
    device='auto',
    patience=20,
    save=True,
)

new_model_path = os.path.join('runs','detect','runs','detect',TRAIN_NAME,'weights','best.pt')
print(f'\nDONE. Model saved at: {new_model_path}')

In [ ]:
# ============================================================
# CELL 7 - Setup Imports and Evaluate on validation set
# ============================================================
import os, glob, random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

TRAIN_NAME = 'wildlife_v2'
data_yaml_path = os.path.join('aug_dataset', 'data.yaml')

YOLO_NAMES = {0:'bull', 1:'buffalo', 2:'lion', 3:'tiger',
               4:'elephant', 5:'zebra', 6:'panda'}

CLASS_COLORS = [
    (0,220,0),(255,140,0),(0,160,255),
    (255,0,255),(255,220,0),(0,255,200),(160,0,255)
]

new_model_path = os.path.join('runs','detect','runs','detect',TRAIN_NAME,'weights','best.pt')
new_model = YOLO(new_model_path)

if os.path.exists(data_yaml_path):
    metrics = new_model.val(data=data_yaml_path, verbose=False)
    print('=== Validation Results ===')
    print(f'mAP50    : {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)')
    print(f'mAP50-95 : {metrics.box.map:.4f}  ({metrics.box.map*100:.1f}%)')
    print(f'Precision: {metrics.box.mp:.4f}')
    print(f'Recall   : {metrics.box.mr:.4f}')
    print()
    if hasattr(metrics.box, 'ap50'):
        print('Per-class AP50:')
        for i, (name, ap) in enumerate(zip(YOLO_NAMES.values(), metrics.box.ap50)):
            bar = '#' * int(ap * 40)
            print(f'  {name:<12} {ap:.3f}  {bar}')
else:
    print(f'Validation dataset not found at {data_yaml_path}, skipping validation.\nModel {new_model_path} loaded successfully.')


In [ ]:
# ============================================================
# CELL 8 — Test on 1 random OldSamples image
#           Re-run to get a different image each time
# ============================================================

new_model_path = os.path.join('runs','detect','runs','detect',TRAIN_NAME,'weights','best.pt')
new_model = YOLO(new_model_path)
print('Classes:', list(new_model.names.values()))

SAMPLE_DIR = 'OldSamples'
all_imgs = []
for ext in ('*.jpg','*.jpeg','*.png'):
    all_imgs.extend(glob.glob(os.path.join(SAMPLE_DIR, ext)))

if len(all_imgs) > 5:
    all_imgs = random.sample(all_imgs, 5)

CONF = 0.20
random.seed()
chosen = random.choice(all_imgs)
fname  = os.path.basename(chosen)
print(f'Image: {fname}')

img = cv2.imread(chosen)
res = new_model(img, conf=CONF, verbose=False)[0]

ann = img.copy()
dets = []
if res.boxes is not None:
    for box in res.boxes:
        x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
        cls_id = int(box.cls[0])
        conf_  = float(box.conf[0])
        name_  = res.names[cls_id]
        color  = CLASS_COLORS[cls_id]
        dets.append(f'{name_} ({conf_:.2f})')
        cv2.rectangle(ann,(x1,y1),(x2,y2),color,2)
        lbl = f'{name_} {conf_:.2f}'
        (lw,lh),base = cv2.getTextSize(lbl,cv2.FONT_HERSHEY_SIMPLEX,0.65,2)
        cv2.rectangle(ann,(x1,y1-lh-base-8),(x1+lw+4,y1),color,-1)
        cv2.putText(ann,lbl,(x1+2,y1-base-2),cv2.FONT_HERSHEY_SIMPLEX,0.65,(0,0,0),2)

result_str = 'Found: '+', '.join(dets) if dets else 'No animals detected'
print(result_str)

fig, axes = plt.subplots(1,2,figsize=(15,7))
fig.patch.set_facecolor('#0f0f1a')
for ax in axes:
    ax.set_facecolor('#1a1a2e'); ax.axis('off')
axes[0].imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
axes[0].set_title('Original',color='white',fontsize=13,fontweight='bold',pad=10)
axes[1].imshow(cv2.cvtColor(ann,cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Detection\n{result_str}',color='#00ffaa',fontsize=12,fontweight='bold',pad=10)
fig.suptitle(f'Wildlife v2  |  {fname}',color='white',fontsize=13,fontweight='bold',y=1.02)
plt.tight_layout()
os.makedirs('output',exist_ok=True)
out_path = os.path.join('output',f'{os.path.splitext(fname)[0]}_v2.jpg')
plt.savefig(out_path,dpi=150,bbox_inches='tight',facecolor=fig.get_facecolor())
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# ============================================================
# CELL 9 — All OldSamples grid
# ============================================================

n = len(all_imgs)
fig, axes = plt.subplots(n,2,figsize=(15,5*n))
fig.patch.set_facecolor('#0f0f1a')
if n==1: axes=[axes]

detected = 0
for i, path in enumerate(sorted(all_imgs)):
    fname = os.path.basename(path)
    img = cv2.imread(path)
    res = new_model(img, conf=CONF, verbose=False)[0]

    ann = img.copy()
    dets = []
    if res.boxes is not None:
        for box in res.boxes:
            x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
            cls_id = int(box.cls[0])
            cf     = float(box.conf[0])
            nm     = res.names[cls_id]
            clr    = CLASS_COLORS[cls_id]
            dets.append(f'{nm}({cf:.2f})')
            cv2.rectangle(ann,(x1,y1),(x2,y2),clr,2)
            lbl = f'{nm} {cf:.2f}'
            (lw,lh),base = cv2.getTextSize(lbl,cv2.FONT_HERSHEY_SIMPLEX,0.6,2)
            cv2.rectangle(ann,(x1,y1-lh-base-7),(x1+lw+3,y1),clr,-1)
            cv2.putText(ann,lbl,(x1+2,y1-base-2),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,0,0),2)

    for ax in axes[i]:
        ax.set_facecolor('#1a1a2e'); ax.axis('off')
    axes[i][0].imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
    axes[i][0].set_title(fname[:45],color='#aaa',fontsize=8)

    col = '#00ffaa' if dets else '#ff6666'
    result_str = ', '.join(dets) if dets else 'No detection'
    if dets: detected += 1
    axes[i][1].imshow(cv2.cvtColor(ann,cv2.COLOR_BGR2RGB))
    axes[i][1].set_title(result_str,color=col,fontsize=9)

fig.suptitle(f'Wildlife v2  |  {detected}/{n} detected',
             color='white',fontsize=13,fontweight='bold',y=1.005)
plt.tight_layout()
plt.show()
print(f'Result: {detected}/{n} images detected')